# Caminho dos Direitos — rodando no Colab com o Google Drive

**Antes de rodar:** coloque os arquivos numa pasta do seu Drive (hoje: `Meu Drive/Dados/Caminho_dos_direitos`; se mudar, troque `PROJETO` na seção 2). Pode ser de qualquer um destes jeitos:

```
CaminhoDosDireitos/                 CaminhoDosDireitos/
├── simulador/                      ├── simulador.py
│   ├── simulador.py                ├── avaliar.py
│   ├── avaliar.py                  ├── regras.csv
│   ├── regras.csv          OU      ├── casos_teste.json
│   └── casos_teste.json            ├── explicador.py
└── ia/                             ├── base_conhecimento.csv
    ├── explicador.py               ├── anotacoes.py   (opcional, seção 10)
    ├── base_conhecimento.csv       └── whatsapp.py    (opcional, seção 11)
    ├── anotacoes.py
    ├── whatsapp.py
    ├── voz.py
    └── servidor_whatsapp.py
```

Depois: **Ambiente de execução → Executar tudo**.

## 1. Conectar o Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Achar os arquivos do projeto
Se a pasta tiver outro nome, troque `PROJETO` abaixo.

In [2]:
import sys, importlib
from pathlib import Path

PROJETO = Path('/content/drive/MyDrive/Dados/notebook_caminho_dos_direitos')  # <- ajuste se precisar

NECESSARIOS = ['simulador.py', 'avaliar.py', 'regras.csv', 'casos_teste.json',
               'explicador.py', 'base_conhecimento.csv']
assert PROJETO.exists(), f'Pasta não encontrada: {PROJETO}. Confira o nome no Drive.'

local = {}
for nome in NECESSARIOS:
    achados = sorted(PROJETO.rglob(nome))
    if achados:
        local[nome] = achados[0]
        print(f'✓ {nome:24s} {achados[0].relative_to(PROJETO)}')
    else:
        print(f'✗ {nome:24s} NÃO ENCONTRADO')
faltando = [n for n in NECESSARIOS if n not in local]
assert not faltando, f'Faltam arquivos no Drive: {faltando}'

PASTA_SIM = local['simulador.py'].parent
PASTA_IA = local['explicador.py'].parent
for p in (PASTA_IA, PASTA_SIM):  # simulador fica na frente do sys.path
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# recarrega os módulos (útil se você editar os .py no Drive e rodar de novo)
import simulador, avaliar, explicador
for m in (simulador, avaliar, explicador):
    importlib.reload(m)
print('\nMódulos carregados de:', PASTA_SIM, 'e', PASTA_IA)

✓ simulador.py             simulador.py
✓ avaliar.py               avaliar.py
✓ regras.csv               regras.csv
✓ casos_teste.json         casos_teste.json
✓ explicador.py            explicador.py
✓ base_conhecimento.csv    base_conhecimento.csv

Módulos carregados de: /content/drive/MyDrive/Dados/notebook_caminho_dos_direitos e /content/drive/MyDrive/Dados/notebook_caminho_dos_direitos


## 3. Motor de regras — persona Dona Cida

In [3]:
from simulador import simular, imprimir, DONA_CIDA

sim = simular(DONA_CIDA)
imprimir(sim)

Prioridade: MÉDIA (8 pontos)

🔴 Travado por documento
  • Documentação básica [TRAVADO_DOC · DOC02]: Alguém da família não tem CPF. O CPF é obrigatório para todos no CadÚnico e para quase todos os benefícios.
      → Tirar o CPF (gratuito). | Onde: Agências dos Correios, Caixa, Banco do Brasil ou Receita Federal | Levar: Certidão de nascimento ou casamento e documento com foto (se tiver)
  • Farmácia Popular [TRAVADO_DOC · FPO01]: Alguém precisa de remédio de uso contínuo ou fralda geriátrica, mas a retirada exige CPF e documento com foto.
      → Tirar o CPF. | Onde: Correios/Caixa/BB | Levar: Certidão de nascimento

🟢 Já recebe ou está em dia
  • CadÚnico [OK · CAD04]: CadÚnico em dia.
  • BPC [JA_RECEBE · BPC02]: Alguém da família já recebe BPC.
      → Manter o CadÚnico atualizado e a biometria em dia. | Onde: — | Levar: —
  • Tarifa Social de Energia [JA_RECEBE · TSE02]: A família já tem Tarifa Social.



## 4. Testes — 30 casos rotulados à mão
Gera `resultado_testes.csv` na mesma pasta do simulador, no Drive.

In [4]:
import pandas as pd

avaliar.main()
df = pd.read_csv(PASTA_SIM / 'resultado_testes.csv', sep=';')
display(df.pivot(index='caso', columns='programa', values='obtido'))

Casos testados: 30  |  Decisões (caso × programa): 210
Acerto por decisão: 210/210 = 100.0%
Casos 100% corretos: 30/30 = 100.0%
Regras exercitadas pelos testes: 46/47
Regras sem teste: PDM04


programa,BPC,Bolsa Família,CadÚnico,Documentação básica,Farmácia Popular,Pé-de-Meia,Tarifa Social de Energia
caso,,,,,,,
C01,PROVAVEL_DIREITO,PROVAVEL_DIREITO,PROVAVEL_DIREITO,ATUALIZAR,PROVAVEL_DIREITO,PROVAVEL_DIREITO,PROVAVEL_DIREITO
C02,NAO_ELEGIVEL,TRAVADO_DOC,TRAVADO_DOC,TRAVADO_DOC,NAO_ELEGIVEL,NAO_ELEGIVEL,TRAVADO_DOC
C03,NAO_ELEGIVEL,NAO_ELEGIVEL,TRAVADO_DOC,TRAVADO_DOC,NAO_ELEGIVEL,NAO_ELEGIVEL,TRAVADO_DOC
C04,NAO_ELEGIVEL,JA_RECEBE,OK,OK,NAO_ELEGIVEL,NAO_ELEGIVEL,JA_RECEBE
C05,NAO_ELEGIVEL,ALERTA,ATUALIZAR,OK,NAO_ELEGIVEL,NAO_ELEGIVEL,PROVAVEL_DIREITO
C06,PROVAVEL_DIREITO,NAO_ELEGIVEL,PROVAVEL_DIREITO,OK,NAO_ELEGIVEL,NAO_ELEGIVEL,PROVAVEL_DIREITO
C07,VERIFICAR,NAO_ELEGIVEL,OK,OK,NAO_ELEGIVEL,NAO_ELEGIVEL,PROVAVEL_DIREITO
C08,NAO_ELEGIVEL,NAO_ELEGIVEL,NAO_ELEGIVEL,OK,NAO_ELEGIVEL,NAO_ELEGIVEL,NAO_ELEGIVEL
C09,NAO_ELEGIVEL,NAO_ELEGIVEL,OK,OK,NAO_ELEGIVEL,PROVAVEL_DIREITO,PROVAVEL_DIREITO


## 5. IA generativa (RAG) — Plano de Direitos
Sem chave de API, usa o modo offline (modelo de texto). Para usar o LLM, rode a célula 7 antes.

In [5]:
from explicador import BaseConhecimento, explicar

base = BaseConhecimento(PASTA_IA / 'base_conhecimento.csv')
r = explicar(sim, base=base)
print(f"[modo: {r['modo']}]\n")
print(r['texto'])
print('\nFontes recuperadas:')
for i, fonte, url in r['fontes']:
    print(f'  {i} — {fonte}')

[modo: offline (modelo de texto)]

*Seu Plano de Direitos*
Este plano mostra possibilidades. Quem confirma é o órgão responsável.

🔴 Primeiro, os documentos
1. *Documentação básica*: Alguém da família não tem CPF. O CPF é obrigatório para todos no CadÚnico e para quase todos os benefícios.
   O que fazer: Tirar o CPF (gratuito).
   Onde: Agências dos Correios, Caixa, Banco do Brasil ou Receita Federal
   Levar: Certidão de nascimento ou casamento e documento com foto (se tiver)
2. *Farmácia Popular*: Alguém precisa de remédio de uso contínuo ou fralda geriátrica, mas a retirada exige CPF e documento com foto.
   O que fazer: Tirar o CPF.
   Onde: Correios/Caixa/BB
   Levar: Certidão de nascimento

🟢 Já recebe ou está em dia
1. *BPC*: Alguém da família já recebe BPC.
   O que fazer: Manter o CadÚnico atualizado e a biometria em dia.
   Onde: —
   Levar: —
2. *Tarifa Social de Energia*: A família já tem Tarifa Social.


Fontes recuperadas:
  K21 — gov.br — Inscrever no CPF
  K20 — Provim

## 6. Dúvida do agente

In [6]:
pergunta = 'A Dona Cida precisa do CPF para pegar o remédio na Farmácia Popular?'
r2 = explicar(sim, pergunta=pergunta, base=base)
print(f"[modo: {r2['modo']}]\n")
print(r2['texto'])

[modo: offline (modelo de texto)]

Trechos mais próximos da pergunta:
- Beneficiários do Bolsa Família são reconhecidos automaticamente pelo sistema na farmácia, sem cadastro prévio. [Ministério da Saúde via SBT News]
- Os medicamentos da lista do Farmácia Popular são entregues de graça em farmácias credenciadas, com receita médica, CPF e documento com foto. Fraldas geriátricas são para pessoas com 60 anos ou mais e pessoas com deficiência. [Ministério da Saúde]
- A inclusão é automática. Em 2026 o MEC usou os dados do CadÚnico até a data-base de 7 de agosto. Divergência entre os dados da matrícula, do CPF e do CadÚnico pode impedir a identificação do estudante. [MEC via Agência Gov]


## 7. (Opcional) Usar o LLM de verdade
1. Clique no ícone de **chave (Secrets)** na barra lateral esquerda.
2. Adicione `ANTHROPIC_API_KEY` com a sua chave e ative o acesso para este notebook.
3. Rode a célula abaixo e depois as células 5 e 6 de novo.

A célula copia a chave do Secrets para a variável de ambiente `ANTHROPIC_API_KEY`, que é onde a biblioteca `anthropic` procura. Sem isso, o explicador continua no modo offline mesmo com a chave salva.

In [7]:
!pip install -q anthropic
import os
from google.colab import userdata
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    importlib.reload(explicador)
    from explicador import BaseConhecimento, explicar
    print('Chave carregada. Rode as células 5 e 6 de novo para usar o LLM.')
except Exception as e:
    print('Chave não encontrada nos Secrets:', type(e).__name__)

Chave não encontrada nos Secrets: SecretNotFoundError


## 8. Testar um caso seu
A pergunta de renda do app é uma **calculadora**: o agente informa quantas pessoas moram na casa e quanto cada uma ganha (em reais ou em salários mínimos). O app calcula a renda por pessoa e guarda **só a faixa**, nunca o valor. Os degraus seguem as regras dos programas: R$ 218 (Bolsa Família), 1/4 do salário mínimo (BPC), meio salário (CadÚnico) e 1 salário.

Preencha só o que mudar; o resto usa o valor padrão do formulário (`CAMPOS_PADRAO` em `simulador.py`).

In [8]:
SALARIO_MINIMO = 1621.00   # 2026

def faixa_renda(rendas, moradores=None, em_salarios=False):
    """Calculadora de renda do app -> código de faixa usado pelo simulador.
    rendas: renda de cada morador que ganha algo (lista). moradores: total de pessoas na casa
    (inclui quem não tem renda). None = a família preferiu não informar."""
    if rendas is None:
        return 'nao_sei'
    total = sum(r * SALARIO_MINIMO if em_salarios else r for r in rendas)
    por_pessoa = total / max(moradores or len(rendas), 1)
    for limite, cod in [(218, 'ate_218'), (SALARIO_MINIMO / 4, 'ate_405'),
                        (SALARIO_MINIMO / 2, 'ate_810'), (SALARIO_MINIMO, 'ate_1621')]:
        if por_pessoa <= limite:
            return cod
    return 'acima'

# Ex.: 3 pessoas na casa, só uma ganha R$ 1.000 -> R$ 333 por pessoa -> 'ate_405'
meu_caso = {
    'renda_faixa': faixa_renda([1000], moradores=3),
    'idoso_65': 'sim',
    'precisa_remedio_ou_fralda': 'sim',
    'no_cadunico': 'nao',
    'recebe': [],                      # bolsa_familia, bpc, tarifa_social, pe_de_meia
}
print('faixa calculada:', meu_caso['renda_faixa'])
sim_meu = simular(meu_caso)
imprimir(sim_meu)
print('\n' + explicar(sim_meu, base=base)['texto'])

faixa calculada: ate_405
Prioridade: ALTA (10 pontos)

🟡 Provavelmente tem direito / fazer agora
  • CadÚnico [PROVAVEL_DIREITO · CAD03]: A renda por pessoa está dentro do limite do CadÚnico (até meio salário mínimo) e a família não está cadastrada.
      → Agendar a inscrição no CadÚnico. Ele é a porta de entrada para Bolsa Família, BPC, Tarifa Social e Pé-de-Meia. | Onde: CRAS (agendamento online da Prefeitura) | Levar: Documentos de todos os moradores (CPF obrigatório) e comprovante de endereço
  • BPC [PROVAVEL_DIREITO · BPC04]: Há idoso de 65+ ou pessoa com deficiência e a renda por pessoa está até 1/4 do salário mínimo (R$ 405,25 em 2026). Pode ter direito a 1 salário mínimo por mês.
      → CadÚnico atualizado → pedido no Meu INSS ou pelo 135. Pessoa com deficiência passa por perícia médica e avaliação social. | Onde: CRAS e depois INSS (Meu INSS ou 135) | Levar: CPF, documento com foto, CadÚnico atualizado e laudos médicos se for por deficiência
  • Tarifa Social de Energia [PR

## 9. Do Plano de Direitos ao que o agente registra
O simulador **sugere**; o agente **confirma**. Cada sugestão já vem com uma situação proposta, que o agente marca na visita: *já recebe, bloqueado, precisa atualizar, falta documento* ou *ainda não pediu*. Quando o simulador diz `ALERTA` (já recebe, mas algo pode bloquear o benefício), a sugestão é *já recebe* com a coluna `alerta` marcada; *bloqueado* só o agente marca, porque depende do que aparece no gov.br ou no CRAS. Cada linha abaixo é uma linha de `fato_item_plano` (grão: um direito sugerido por caso), a mesma tabela do modelo dimensional do app.

Isso é só demonstração do formato: **não vai para o dashboard**, que usa os dados simulados do notebook `Simulacao_App_v2`.

In [9]:
import pandas as pd

# status do motor de regras -> situação sugerida ao agente (dim_situacao)
SITUACAO_SUGERIDA = {
    'PROVAVEL_DIREITO': 'Ainda não pediu',
    'ATUALIZAR':        'Precisa atualizar',
    'TRAVADO_DOC':      'Falta documento',
    'JA_RECEBE':        'Já recebe',
    'ALERTA':           'Já recebe',   # já recebe, mas com alerta (ex.: cadastro desatualizado pode bloquear)
    'VERIFICAR':        'Agente confirma na visita',
}
# OK e NAO_ELEGIVEL não viram item do plano

def _itens(obj):
    """Acha, dentro do resultado do simulador, os dicionários que têm programa + status."""
    if hasattr(obj, 'to_dict') and not isinstance(obj, dict):
        obj = obj.to_dict('records') if hasattr(obj, 'columns') else obj.to_dict()
    elif hasattr(obj, '__dict__') and not isinstance(obj, (dict, list, tuple)):
        obj = vars(obj)
    if isinstance(obj, dict):
        chaves = {str(k).lower() for k in obj}
        if 'programa' in chaves and ({'status', 'situacao', 'resultado'} & chaves):
            yield obj
        else:
            for v in obj.values():
                yield from _itens(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from _itens(v)

def _campo(item, *nomes):
    low = {str(k).lower(): v for k, v in item.items()}
    return next((low[n] for n in nomes if n in low), None)

def itens_do_plano(sim, id_caso, faixa=None):
    linhas = []
    for it in _itens(sim):
        status = _campo(it, 'status', 'situacao', 'resultado')
        if status not in SITUACAO_SUGERIDA:
            continue
        programa = _campo(it, 'programa')
        sugerida = SITUACAO_SUGERIDA[status]
        if programa == 'Documentação básica' and status == 'ATUALIZAR':
            sugerida = 'Falta documento'          # ex.: alguém sem RG/CIN
        linhas.append({'id_caso': id_caso, 'programa': programa, 'regra': _campo(it, 'regra_id', 'regra', 'id'),
                       'status_simulador': status, 'situacao_sugerida': sugerida, 'alerta': status == 'ALERTA',
                       'situacao_confirmada': None,   # o agente marca no app
                       'faixa_renda': faixa})
    if not linhas:
        print('Não achei os itens no resultado do simulador. Tipo:', type(sim).__name__,
              '| chaves:', list(sim)[:10] if isinstance(sim, dict) else '-')
    return pd.DataFrame(linhas)

display(pd.concat([itens_do_plano(sim, 'Dona Cida'),
                   itens_do_plano(sim_meu, 'Meu caso', meu_caso['renda_faixa'])], ignore_index=True))

,id_caso,programa,regra,status_simulador,situacao_sugerida,alerta,situacao_confirmada,faixa_renda
0,Dona Cida,Documentação básica,DOC02,TRAVADO_DOC,Falta documento,False,None,None
1,Dona Cida,BPC,BPC02,JA_RECEBE,Já recebe,False,None,None
2,Dona Cida,Tarifa Social de Energia,TSE02,JA_RECEBE,Já recebe,False,None,None
3,Dona Cida,Farmácia Popular,FPO01,TRAVADO_DOC,Falta documento,False,None,None
4,Meu caso,CadÚnico,CAD03,PROVAVEL_DIREITO,Ainda não pediu,False,None,ate_405
5,Meu caso,BPC,BPC04,PROVAVEL_DIREITO,Ainda não pediu,False,None,ate_405
6,Meu caso,Tarifa Social de Energia,TSE05,PROVAVEL_DIREITO,Ainda não pediu,False,None,ate_405
7,Meu caso,Farmácia Popular,FPO02,PROVAVEL_DIREITO,Ainda não pediu,False,None,ate_405


## 10. Anotação por voz com IA
Na visita, o agente **fala** a anotação pelo microfone (ou envia um áudio, ou digita). O áudio é transcrito **no próprio servidor** (Whisper) e apagado. Antes de qualquer IA ver o texto, o módulo **remove os dados clínicos** (pressão, peso, altura, glicemia, nome de doença ou remédio) e avisa: *registre no e-SUS*. Depois **organiza** tudo num resumo da visita (situação relatada, prioridade, próximos passos e pendências, como laudo que falta) e sugere 5 marcações de direitos, que o agente confirma:

| Marcação | O que muda no caso |
|---|---|
| Dificuldade de ir ao posto | Visita domiciliar ou atendimento acompanhado |
| Pessoa com deficiência (tem laudo?) | Conferir BPC; laudo na lista do que levar |
| Precisa de atendimento médico | Encaminhar para a UBS |
| Uso contínuo de remédio ou fralda | Conferir Farmácia Popular |
| Situação de urgência (comida, moradia, luz) | Prioridade e aviso ao coordenador |

O app guarda **só as marcações confirmadas** (`fato_necessidade_caso`): nem o texto, nem o áudio, nem o dado clínico. Sem chave de API, funciona só com as regras; com a chave (seção 7), o Claude ajuda a achar o que foi dito de outro jeito e recebe apenas o texto já filtrado.

In [10]:
try:
    import anotacoes; importlib.reload(anotacoes)
    from anotacoes import organizar, anotar_por_voz, anotar_pelo_microfone
except ImportError:
    raise ImportError('Coloque anotacoes.py na pasta ia/ do Drive e rode a seção 2 de novo.')

# Opção A: anotação digitada (rápida para a demonstração)
nota = ('Visitei a Dona Cida. Ela não consegue ir sozinha ao Poupatempo, tem dificuldade para andar. '
        'O filho trabalha de manhã e só pode acompanhar à tarde. Pressão 16 por 10, toma losartana. '
        'Ela precisa passar no médico logo.')
res_nota = organizar(nota)

# Opção B: falar pelo microfone (mude para True; clique em Permitir, fale e depois em "Parar e organizar")
USAR_MICROFONE = False
if USAR_MICROFONE:
    !pip install -q faster-whisper
    res_nota = anotar_pelo_microfone(max_segundos=180)   # grava, transcreve, apaga o áudio e organiza

# Opção C: áudio gravado no celular (mude para True e envie o arquivo .ogg/.m4a/.mp3)
USAR_AUDIO = False
if USAR_AUDIO:
    !pip install -q faster-whisper
    from google.colab import files
    enviado = files.upload()
    res_nota = anotar_por_voz(next(iter(enviado)))    # transcreve, apaga o áudio e organiza

res_nota.mostrar()

# O agente confirma as marcações; elas preenchem o formulário e o plano é refeito
for m in res_nota.marcacoes:
    m.confirmada = True
print('\nRespostas que as marcações preenchem:', res_nota.para_formulario(so_confirmadas=True))
imprimir(simular({**DONA_CIDA, **res_nota.para_formulario(so_confirmadas=True)}))
print('\nO que fica salvo (fato_necessidade_caso):')
display(pd.DataFrame(res_nota.para_salvar('Dona Cida', '2026-11-12')))

[modo: regras (offline)]
Texto filtrado (só na tela, não é salvo): Visitei a Dona Cida. Ela não consegue ir sozinha ao Poupatempo, tem dificuldade para andar. O filho trabalha de manhã e só pode acompanhar à tarde. [dado de saúde: registrar no e-SUS] toma [dado de saúde: registrar no e-SUS] . Ela precisa passar no médico logo.
⚠️  A anotação tinha pressão, nome de remédio. Isso não fica no app: registre no e-SUS.

📋 RESUMO DA VISITA
  Situação relatada:
    • Visitei a Dona Cida
    • Ela não consegue ir sozinha ao Poupatempo, tem dificuldade para andar
    • O filho trabalha de manhã e só pode acompanhar à tarde
    • Ela precisa passar no médico logo
  Próximos passos:
    1. Encaminhar para a UBS (sem registrar o motivo aqui)
    2. Encaminhamento por visita domiciliar ou atendimento acompanhado
    3. Conferir Farmácia Popular
    4. Registrar no e-SUS: pressão, nome de remédio
  Pendências (perguntar ou providenciar):
    • Combinar quem acompanha ou se o atendimento será em casa


,id_caso,data,categoria,tem_laudo,origem
0,Dona Cida,2026-11-12,dificuldade_ir_ao_posto,None,regra
1,Dona Cida,2026-11-12,precisa_atendimento_medico,None,regra
2,Dona Cida,2026-11-12,uso_continuo_remedio_ou_fralda,None,"regra (remédio citado, nome não guardado)"


## 11. Lembrete por WhatsApp em texto e em áudio
Recurso de acessibilidade: para quem não lê bem, enxerga mal ou prefere ouvir.

1. O agente salva o encaminhamento → a pessoa recebe o lembrete **em texto** com o botão **"Ouvir em áudio"**.
2. A pessoa toca no botão → recebe o mesmo lembrete como **mensagem de voz**.

Por que nessa ordem: fora da janela de 24 horas o WhatsApp só permite mensagem-modelo aprovada (texto). Quando a pessoa toca no botão, a conversa abre e o áudio pode ir.

Sem `WHATSAPP_TOKEN` e `WHATSAPP_PHONE_ID` nos Secrets, a célula roda em **modo demonstração**: nada é enviado, mas o áudio é gerado e toca aqui. Para enviar de verdade, use o número de teste gratuito da Meta e cadastre a mensagem-modelo `TEMPLATE_LEMBRETE` de `whatsapp.py`.

A voz do áudio é **neural** (`voz.py`, vozes Francisca/Antonio da Microsoft), um pouco mais devagar que o normal e com datas, horários e siglas lidos por extenso ("18 de novembro", "9 horas", "cê pê éfe"). Se ela falhar, volta sozinha para a voz do gTTS.

In [11]:
!pip install -q gTTS edge-tts
try:
    import whatsapp; importlib.reload(whatsapp)
    from whatsapp import WhatsApp, Lembrete, tratar_webhook, toque_simulado, falar
except ImportError:
    raise ImportError('Coloque whatsapp.py na pasta ia/ do Drive e rode a seção 2 de novo.')
from IPython.display import Audio, display
try:   # voz neural (mais humana) no lugar do gTTS; o resto do whatsapp.py não muda
    import voz; importlib.reload(voz)
    whatsapp.falar = falar = voz.falar
    print('Voz: neural (voz.py)')
except ImportError:
    print('voz.py não está no Drive: usando a voz do gTTS')

import os
try:   # chaves opcionais nos Secrets do Colab
    from google.colab import userdata
    os.environ['WHATSAPP_TOKEN'] = userdata.get('WHATSAPP_TOKEN')
    os.environ['WHATSAPP_PHONE_ID'] = userdata.get('WHATSAPP_PHONE_ID')
except Exception:
    pass

wa = WhatsApp()
print('Modo demonstração (nada é enviado):', wa.demonstracao)
TELEFONE = '(11) 98765-4321'   # no modo real, um dos números cadastrados no app de teste da Meta

# Mesmo encaminhamento da tela 07 do protótipo
lembrete = Lembrete('E0142', 'Dona Cida', "Poupatempo M'Boi Mirim", '18/11', '9h',
                    'certidão de nascimento, RG e comprovante de residência')
wa.enviar_lembrete(TELEFONE, lembrete)
print('1) Texto enviado:', lembrete.texto, '[botão: Ouvir em áudio]')

# 2) A pessoa toca no botão: o WhatsApp avisa o webhook, que responde com o áudio
tratar_webhook(toque_simulado(TELEFONE, 'E0142'), wa, {'E0142': lembrete}.get)
print('2) Áudio enviado como mensagem de voz:')
display(Audio('lembrete_E0142.ogg'))

# Extra: o Plano de Direitos inteiro em áudio (o botão "Ouvir em voz alta" da tela 06)
print('Plano de Direitos em áudio:')
display(Audio(str(falar(explicar(sim, base=base)['texto'], 'plano_dona_cida.ogg'))))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
Voz: neural (voz.py)
Modo demonstração (nada é enviado): True
1) Texto enviado: Olá, Dona Cida! Seu atendimento no Poupatempo M'Boi Mirim é dia 18/11 às 9h. Leve: certidão de nascimento, RG e comprovante de residência. [botão: Ouvir em áudio]
2) Áudio enviado como mensagem de voz:


Plano de Direitos em áudio:


### 11b. Comparar as vozes
A mesma mensagem nas três vozes. O grupo escolhe a que fica melhor para a apresentação (padrão: `feminina`).

In [12]:
import voz; importlib.reload(voz)
from gtts import gTTS
gTTS(lembrete.texto, lang='pt', tld='com.br').save('voz_gtts.mp3')
print('Antes (gTTS):'); display(Audio('voz_gtts.mp3'))
for nome in ['feminina', 'feminina2', 'masculina']:
    arq = voz.falar(lembrete.texto, f'voz_{nome}.ogg', voz=nome)
    print(f'{nome} ({voz.VOZES[nome]}) — motor: {voz.falar.ultimo_motor}')
    display(Audio(str(arq)))

Antes (gTTS):


feminina (pt-BR-FranciscaNeural) — motor: neural (edge-tts)


feminina2 (pt-BR-ThalitaNeural) — motor: neural (edge-tts)


masculina (pt-BR-AntonioNeural) — motor: neural (edge-tts)


## 12. WhatsApp de verdade (servidor com webhook)
A seção 11 simula. Para mandar e receber no WhatsApp de verdade, abra o notebook **`servidor_whatsapp_colab.ipynb`**: ele sobe o servidor (`servidor_whatsapp.py`, feito pelo manual da Cloud API) aqui no Colab, cria um endereço público HTTPS e mostra o que colar no painel da Meta.